# 04 - Fairness: group diagnostics

The target is a lender's recorded decision, not creditworthiness or default. So every number below says how the model behaves against past decisions, group by group. Agreeing with those decisions is not the same as being right about the applicant. All numbers are associations, not causes.

This notebook only reads predictions already saved. It trains nothing. To study another model, change RUN_ID and run it again.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from compare import weighted_auc
from data import load

RUN_ID = "20261014T115111Z_918eb62b"   # LightGBM + CatBoost county, 50/50 log-odds: best model so far
N_BOOT, BOOT_SEED = 1000, 0            # same as src/compare.py
MIN_N = 500                            # under this, a group is shown but not read

train, _ = load()
oof = pd.read_parquet(ROOT / "oof" / f"{RUN_ID}.parquet")
df = train.merge(oof[["id", "pred"]], on="id", validate="1:1")
assert (df.accepted == df.merge(oof, on="id").accepted_y).all()
y, p = df.accepted.to_numpy(), df.pred.to_numpy()
print(len(df), "rows   pooled AUC", round(roc_auc_score(y, p), 5), "  accepted", round(y.mean(), 4), "  mean prediction", round(p.mean(), 4))

105000 rows   pooled AUC 0.92   accepted 0.8499   mean prediction 0.8514


## Choices

Groups: "derived_race", "derived_ethnicity" and "derived_sex" (they summarise applicant and co-applicant together), and "applicant_age". They are the official HMDA summaries, so a reader can check them. "Joint", "Not Available" and "Free Form Text Only" are kept: dropping them would hide a large share of the rows.

Reference group: the largest group with a single declared value (White, Not Hispanic or Latino, Male, 25-34). A gap is measured against it because a gap needs a base, not because that group is the norm. "Joint" is larger for sex, but it describes a couple, not a person.

Two cut-offs: the metric is AUC, so there is no real decision threshold and any choice is a convention. We take 0.50 (the plain reading of a probability) and the cut-off that accepts as many rows as were really accepted (0.85 of them). The same cut-off applies to every group: separate cut-offs would be a different model.

Positive class = accepted. So TPR = share of accepted rows the model also accepts, FPR = share of denied rows the model accepts anyway. Here "error" means "disagrees with the lender".

Intervals: bootstrap of rows inside each group, 1000 resamples, percentile 95%. The group sizes stay fixed. A gap's interval takes the two groups' resamples pair by pair. These intervals cover the sampling of rows, not the training of the model, and they are not corrected for reading many groups at once.

"AUC without leak rows" recomputes AUC without the rows where a confirmed leak is active (preapproval = 1, initially_payable_to_institution = 3). The model still learned from those columns. It is only a first look: the honest number comes from the model trained without them.

In [2]:
GROUPS = {"derived_race": "White", "derived_ethnicity": "Not Hispanic or Latino",
          "derived_sex": "Male", "applicant_age": "25-34"}
CUTS = {"0.50": 0.5, "same rate": float(np.quantile(p, 1 - y.mean()))}
leak = (df.preapproval.astype("string") == "1") | (df.initially_payable_to_institution.astype("string") == "3")
keep = ~leak.to_numpy()
print("cut-offs", {k: round(v, 4) for k, v in CUTS.items()}, "  leak rows", int(leak.sum()))

STATS = ["observed", "predicted", "pred_minus_obs", "auc", "auc_no_leak"]
for c in CUTS:
    STATS += [f"accept {c}", f"tpr {c}", f"fpr {c}"]


def group_stats(y, p, keep):
    """Return f(w): every statistic of one group for row weights w (a resample = how many times each row is drawn)."""
    auc, auc_kept = weighted_auc(y, p), weighted_auc(y[keep], p[keep])
    above = {c: p >= t for c, t in CUTS.items()}

    def f(w):
        total, pos, neg = w.sum(), (w * y).sum(), (w * (1 - y)).sum()
        obs, pred = pos / total, (w * p).sum() / total
        out = [obs, pred, pred - obs, auc(w), auc_kept(w[keep])]
        for a in above.values():
            out += [(w * a).sum() / total, (w * y * a).sum() / pos, (w * (1 - y) * a).sum() / neg]
        return np.array(out)

    return f


def diagnostics(attr):
    rng = np.random.default_rng(BOOT_SEED)
    labels = df[attr].astype("string").fillna("(blank)").to_numpy()
    point, draws, size = {}, {}, {}
    for level in pd.Series(labels).value_counts().index:
        i = np.flatnonzero(labels == level)
        f, n = group_stats(y[i], p[i], keep[i]), len(i)
        point[level] = f(np.ones(n))
        draws[level] = np.array([f(np.bincount(rng.integers(0, n, n), minlength=n).astype(float)) for _ in range(N_BOOT)])
        size[level] = (n, int((y[i] == 0).sum()))
    return point, draws, size


np.seterr(all="ignore")   # very small groups can draw a resample with no denied row: that draw is NaN
results = {attr: diagnostics(attr) for attr in GROUPS}

cut-offs {'0.50': 0.5, 'same rate': 0.779}   leak rows 6279


In [3]:
# Checks: the weighted formulas must give back the plain ones.
i = np.flatnonzero(df.derived_race.astype("string") == "Black or African American")
pt = results["derived_race"][0]["Black or African American"]
d = dict(zip(STATS, pt))
assert abs(d["auc"] - roc_auc_score(y[i], p[i])) < 1e-9
assert abs(d["auc_no_leak"] - roc_auc_score(y[i][keep[i]], p[i][keep[i]])) < 1e-9
assert abs(d["observed"] - y[i].mean()) < 1e-12
t = CUTS["same rate"]
assert abs(d["tpr same rate"] - (p[i][y[i] == 1] >= t).mean()) < 1e-12
assert abs(d["fpr same rate"] - (p[i][y[i] == 0] >= t).mean()) < 1e-12
print("checks passed")

checks passed


## Calibration and ranking inside each group

Observed = share accepted. Predicted = mean predicted probability. A positive gap means the model expects more acceptances than the lender gave. Groups under 500 rows are marked and not read.

In [4]:
def ci(a, digits=3):
    lo, hi = np.nanquantile(a, [0.025, 0.975])
    return f"[{lo:.{digits}f}, {hi:.{digits}f}]"


def table_one(attr):
    point, draws, size = results[attr]
    rows = []
    for level, pt in point.items():
        n, denied = size[level]
        row = {"group": level, "n": n, "denied": denied}
        if n < MIN_N:
            rows.append({**row, "read": "no, n < 500"})
            continue
        d, b = dict(zip(STATS, pt)), draws[level]
        col = lambda s: b[:, STATS.index(s)]
        row |= {"observed": f"{d['observed']:.3f} {ci(col('observed'))}",
                "predicted": f"{d['predicted']:.3f}",
                "pred - obs": f"{d['pred_minus_obs']:+.3f} {ci(col('pred_minus_obs'))}",
                "AUC": f"{d['auc']:.3f} {ci(col('auc'))}",
                "AUC without leak rows": f"{d['auc_no_leak']:.3f} {ci(col('auc_no_leak'))}", "read": "yes"}
        rows.append(row)
    return pd.DataFrame(rows).fillna("")


for attr in GROUPS:
    print(attr); print(table_one(attr).to_string(index=False)); print()

derived_race
                                    group     n  denied             observed predicted             pred - obs                  AUC AUC without leak rows        read
                                    White 66107    8721 0.868 [0.865, 0.871]     0.869 +0.001 [-0.001, 0.002] 0.921 [0.917, 0.924]  0.907 [0.903, 0.911]         yes
                       Race Not Available 17872    3082 0.828 [0.822, 0.833]     0.830 +0.002 [-0.001, 0.006] 0.922 [0.915, 0.928]  0.908 [0.900, 0.915]         yes
                Black or African American  8991    2349 0.739 [0.730, 0.748]     0.743 +0.004 [-0.001, 0.010] 0.909 [0.901, 0.917]  0.896 [0.886, 0.904]         yes
                                    Asian  8009     888 0.889 [0.882, 0.896]     0.893 +0.004 [-0.001, 0.009] 0.880 [0.865, 0.894]  0.853 [0.834, 0.870]         yes
                                    Joint  2716     347 0.872 [0.859, 0.884]     0.872 -0.000 [-0.008, 0.008] 0.909 [0.887, 0.932]  0.893 [0.868, 0.919]         y

## Decisions at a fixed cut-off, gaps against the reference group

For each group: share the model accepts, TPR and FPR, then the two gaps against the reference group with their 95% intervals. A gap whose interval excludes 0 is a measured difference on these rows; it does not say why.

In [5]:
def table_two(attr, cut):
    point, draws, size = results[attr]
    ref, rows = GROUPS[attr], []
    for level, pt in point.items():
        n = size[level][0]
        row = {"group": level + (" (reference)" if level == ref else ""), "n": n}
        if n < MIN_N:
            rows.append({**row, "read": "no, n < 500"})
            continue
        d = dict(zip(STATS, pt))
        row |= {"accepts": f"{d[f'accept {cut}']:.3f}", "TPR": f"{d[f'tpr {cut}']:.3f}", "FPR": f"{d[f'fpr {cut}']:.3f}"}
        if level != ref:
            for s, name in ((f"tpr {cut}", "TPR gap"), (f"fpr {cut}", "FPR gap")):
                k = STATS.index(s)
                gap = pt[k] - point[ref][k]
                row[name] = f"{gap:+.3f} {ci(draws[level][:, k] - draws[ref][:, k])}"
        rows.append({**row, "read": "yes"})
    return pd.DataFrame(rows).fillna("")


for cut, t in CUTS.items():
    for attr in GROUPS:
        print(f"{attr}, cut-off {cut} ({t:.4f})"); print(table_two(attr, cut).to_string(index=False)); print()

derived_race, cut-off 0.50 (0.5000)
                                    group     n accepts   TPR   FPR        read                 TPR gap                 FPR gap
                        White (reference) 66107   0.903 0.986 0.355         yes                                                
                       Race Not Available 17872   0.869 0.983 0.321         yes -0.004 [-0.006, -0.002] -0.034 [-0.054, -0.015]
                Black or African American  8991   0.794 0.972 0.293         yes -0.015 [-0.019, -0.011] -0.062 [-0.082, -0.041]
                                    Asian  8009   0.931 0.993 0.430         yes   +0.007 [0.005, 0.009]   +0.075 [0.041, 0.108]
                                    Joint  2716   0.902 0.985 0.337         yes  -0.002 [-0.007, 0.003]  -0.018 [-0.067, 0.032]
         American Indian or Alaska Native   810   0.752 0.961 0.195         yes -0.025 [-0.042, -0.010] -0.161 [-0.210, -0.107]
                 2 or more minority races   278                     

## Tables for the report

The cell below prints the three tables of "reports/fairness.md" in markdown, so the report is copied from here and never typed by hand. Their reading is in the report.

In [6]:
SHORT = {"derived_race": "race", "derived_ethnicity": "ethnicity", "derived_sex": "sex", "applicant_age": "age"}


def markdown(table):
    lines = ["| " + " | ".join(table.columns) + " |", "|" + "---|" * len(table.columns)]
    return "\n".join(lines + ["| " + " | ".join(str(v) for v in row) + " |" for row in table.to_numpy()])


def report_table(make, drop):
    parts = [make(attr).assign(field=SHORT[attr]) for attr in GROUPS]
    t = pd.concat(parts).fillna("")
    first = next(c for c in t.columns if c not in ("group", "n", "denied", "read", "field"))
    t[first] = t[first].where(t.read == "yes", "not read")
    return t[["field"] + [c for c in t.columns if c not in ("field", "read", *drop)]]


print(markdown(report_table(table_one, ["predicted"])))
for cut in CUTS:
    print(f"\n<cut {cut}>\n" + markdown(report_table(lambda a: table_two(a, cut), [])))

| field | group | n | denied | observed | pred - obs | AUC | AUC without leak rows |
|---|---|---|---|---|---|---|---|
| race | White | 66107 | 8721 | 0.868 [0.865, 0.871] | +0.001 [-0.001, 0.002] | 0.921 [0.917, 0.924] | 0.907 [0.903, 0.911] |
| race | Race Not Available | 17872 | 3082 | 0.828 [0.822, 0.833] | +0.002 [-0.001, 0.006] | 0.922 [0.915, 0.928] | 0.908 [0.900, 0.915] |
| race | Black or African American | 8991 | 2349 | 0.739 [0.730, 0.748] | +0.004 [-0.001, 0.010] | 0.909 [0.901, 0.917] | 0.896 [0.886, 0.904] |
| race | Asian | 8009 | 888 | 0.889 [0.882, 0.896] | +0.004 [-0.001, 0.009] | 0.880 [0.865, 0.894] | 0.853 [0.834, 0.870] |
| race | Joint | 2716 | 347 | 0.872 [0.859, 0.884] | -0.000 [-0.008, 0.008] | 0.909 [0.887, 0.932] | 0.893 [0.868, 0.919] |
| race | American Indian or Alaska Native | 810 | 221 | 0.727 [0.696, 0.758] | -0.009 [-0.027, 0.009] | 0.939 [0.917, 0.960] | 0.933 [0.908, 0.956] |
| race | 2 or more minority races | 278 | 87 | not read |  |  |  |
| race